# 01. KOSPI OHLCV 데이터 수집
pykrx로 종목별 주가 데이터를 수집하여 `data/ohlcv.csv`로 저장

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pykrx

In [ ]:
# ── 설정 (본인 Drive 경로에 맞게 수정) ───────────────────────────────────────
BASE_DIR          = "/content/drive/MyDrive/term_project"
DATA_DIR          = f"{BASE_DIR}/data"
STOCK_CODES_PATH  = f"{BASE_DIR}/kospi_stock_codes.csv"
START_DATE        = "20060101"
END_DATE          = "20260401"

import os
os.makedirs(DATA_DIR, exist_ok=True)
print(f"DATA_DIR: {DATA_DIR}")

In [ ]:
import pandas as pd
from pykrx import stock
import time

In [ ]:
def fetch_ohlcv(code: str, start: str, end: str) -> pd.DataFrame:
    df = stock.get_market_ohlcv_by_date(start, end, code)
    df = df.reset_index()
    df.rename(columns={
        "날짜": "date", "시가": "open", "고가": "high",
        "저가": "low", "종가": "close", "거래량": "volume",
    }, inplace=True)
    df["code"] = code
    return df

In [ ]:
# ── 종목 코드 로드 ─────────────────────────────────────────────────────────────
# kospi_stock_codes.csv 를 Drive에 업로드해야 합니다
df_codes = pd.read_csv(STOCK_CODES_PATH, dtype={"종목코드": str})
codes = df_codes["종목코드"].str.zfill(6).tolist()
print(f"종목 수: {len(codes)}")

In [ ]:
# ── OHLCV 수집 ────────────────────────────────────────────────────────────────
all_data = []
total = len(codes)

for i, code in enumerate(codes, 1):
    print(f"[{i}/{total}] {code} 수집 중...", end=" ", flush=True)
    try:
        df = fetch_ohlcv(code, START_DATE, END_DATE)
        all_data.append(df)
        print(f"{len(df)}행")
    except Exception as e:
        print(f"실패: {e}")
    time.sleep(0.3)

In [ ]:
# ── 저장 ──────────────────────────────────────────────────────────────────────
result = pd.concat(all_data, ignore_index=True)
output_path = f"{DATA_DIR}/ohlcv.csv"
result.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path} ({len(result)}행)")
result.head()